# Ablation ESCO trên resume-job-fit (cnamuangtoun) – cấu hình chính pair_skip_mlp_specfeat

**Settings → Accelerator: GPU T4 x2 hoặc P100** (GPU chỉ để mã hóa BGE-M3 lần đầu; huấn luyện GNN chạy CPU), **Internet: On**. Không cần khóa API.

Bấm **Save Version → Save & Run All** (khoảng 3–4 giờ). Xong thì tải 3 file `ext_cna_req_jd_abl2_esco.csv`, `ext_cna_req_jd_abl2_esco_val.csv`, `ext_cna_maxsim_jd_abl2_esco.csv` ở tab Output, đặt vào `report/results/` trên máy.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv; nproc
!git clone -q -b bang-branch https://github.com/hoangphungvan/doantotnghiep.git
%cd doantotnghiep
!git log --oneline -1

In [ ]:
# Chỉ cài phần còn thiếu, KHÔNG cài lại torch (giữ bản CUDA của Kaggle)
!pip install -q torch_geometric python-dotenv openai pymupdf
import torch, sentence_transformers; print(torch.__version__, torch.cuda.is_available(), sentence_transformers.__version__)

In [ ]:
# Bộ dữ liệu không nằm trong repo: tải train.csv/test.csv gốc từ HuggingFace
from huggingface_hub import snapshot_download
snapshot_download('cnamuangtoun/resume-job-description-fit', repo_type='dataset', local_dir='data/raw/cnamuangtoun')
!ls -la data/raw/cnamuangtoun

In [ ]:
# 5 seed x 5 fold theo JD, mục kỹ năng chuẩn hóa ESCO ngưỡng 0,85 (bảng ghép data/cache/esco_map_BAAI_bge_m3.json có trong repo)
!mkdir -p /kaggle/working
!REQ_ESCO=0.85 python -X utf8 src/modeling/train_ext.py --protocol jd --configs pair_skip_mlp_specfeat --tag _jd_abl2_esco; cp report/results/ext_cna_*abl2_esco*.csv /kaggle/working/

In [ ]:
!cp report/results/ext_cna_*abl2_esco*.csv /kaggle/working/ 2>/dev/null; ls -la /kaggle/working/*.csv